# WhisperSubs Colab
Japanese audio â†’ casual Indonesian SRT using kotoba-whisper-v2.0 (float16) + GPT-4o

> **Before running:** Runtime â†’ Change runtime type â†’ T4 GPU
>
> **Workflow:** Run all cells top to bottom. Enter API key when prompted. Upload audio. Download SRT.

In [ ]:
# Cell 1: Install dependencies
!pip install -q faster-whisper openai
print('Dependencies installed.')

In [ ]:
# Cell 2: Configuration — run this once per session
import getpass
import os

OPENAI_API_KEY = getpass.getpass("OpenAI API Key: ")
GPT_MODEL = "gpt-4o"
WHISPER_MODEL = "jctv-tech/kotoba-whisper-v21-ct2"
BATCH_SIZE = 5

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"Config set.")
print(f"  GPT model   : {GPT_MODEL}")
print(f"  Whisper model: {WHISPER_MODEL}")
print(f"  Batch size  : {BATCH_SIZE} segments per GPT call")

In [ ]:
# Cell 3: Upload audio file
from google.colab import files

print("Select your audio file (MP3, WAV, M4A, FLAC, etc.):")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Re-run this cell and select a file.")

audio_filename = list(uploaded.keys())[0]
file_size_mb = len(uploaded[audio_filename]) / 1024 / 1024
print(f"
Ready: {audio_filename} ({file_size_mb:.1f} MB)")

In [ ]:
# Cell 4: Transcribe audio
# First run: downloads model to Colab (~3-5 min). Subsequent cells in same session use cached model.
from faster_whisper import WhisperModel

print(f"Loading model: {WHISPER_MODEL} on T4 GPU (float16)...")
print("(First run downloads ~3GB — takes 3-5 min. Cached for this session.)")

model = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")

print("Transcribing...")
segments_iter, info = model.transcribe(
    audio_filename,
    language="ja",
    beam_size=5,
    condition_on_previous_text=False,
    no_speech_threshold=0.6,
    compression_ratio_threshold=2.4,
    log_prob_threshold=-1.0,
    vad_filter=True,
    vad_parameters={
        "threshold": 0.5,
        "min_silence_duration_ms": 1000,
        "speech_pad_ms": 200,
    },
)

print(f"Detected language: {info.language} (confidence: {info.language_probability:.2f})")

segments = []
for seg in segments_iter:
    segments.append({
        "start": seg.start,
        "end": seg.end,
        "text": seg.text.strip(),
    })

print(f"\nTranscription complete! Total segments: {len(segments)}")
if segments:
    print(f"  First: [{segments[0]['start']:.2f}s → {segments[0]['end']:.2f}s] {segments[0]['text'][:80]}")
    print(f"  Last:  [{segments[-1]['start']:.2f}s → {segments[-1]['end']:.2f}s] {segments[-1]['text'][:80]}")